# Interaction Laws — Python Baseline for ludoSpring

**SPDX-License-Identifier: AGPL-3.0-or-later**

Reference implementations of Fitts's law, Hick's law, and steering law.
These produce the ground truth that the Rust implementation must match.

### References
- Fitts, P.M. (1954). "The information capacity of the human motor system."
- MacKenzie, I.S. (1992). "Fitts' law as a research and design tool in HCI."
- Hick, W.E. (1952). "On the rate of gain of information."
- Hyman, R. (1953). "Stimulus information as a determinant of reaction time."
- Accot, J. & Zhai, S. (1997). "Beyond Fitts' law: models for trajectory-based HCI tasks."

In [ ]:
import math

# Constants matching ludoSpring tolerances/interaction.rs
FITTS_A_MOUSE_MS = 50.0
FITTS_B_MOUSE_MS = 150.0
HICK_A_MS = 200.0
HICK_B_MS = 150.0

## Fitts's Law

Shannon formulation: $MT = a + b \cdot \log_2\left(\frac{2D}{W} + 1\right)$

In [ ]:
def fitts_movement_time(distance, width, a, b):
    """Shannon formulation: MT = a + b * log2(2D/W + 1)"""
    return a + b * math.log2(2.0 * distance / width + 1.0)

def fitts_index_of_difficulty(distance, width):
    """ID = log2(2D/W + 1)"""
    return math.log2(2.0 * distance / width + 1.0)

# Canonical test case: D=100, W=10
mt = fitts_movement_time(100.0, 10.0, FITTS_A_MOUSE_MS, FITTS_B_MOUSE_MS)
id_val = fitts_index_of_difficulty(100.0, 10.0)
print(f"Fitts MT (D=100, W=10): {mt:.10f} ms")
print(f"Fitts ID (D=100, W=10): {id_val:.10f} bits")

In [ ]:
# Game scenarios: Doom-style aiming at targets of varying distance/size
scenarios = [
    ("close_barrel", 50.0, 30.0),
    ("medium_imp", 150.0, 20.0),
    ("far_cacodemon", 300.0, 15.0),
    ("sniper_far_tiny", 400.0, 5.0),
]

print(f"{'Scenario':<20} {'D':>6} {'W':>6} {'MT (ms)':>10} {'ID (bits)':>10}")
print("-" * 56)
for name, d, w in scenarios:
    mt = fitts_movement_time(d, w, FITTS_A_MOUSE_MS, FITTS_B_MOUSE_MS)
    id_v = fitts_index_of_difficulty(d, w)
    print(f"{name:<20} {d:>6.0f} {w:>6.0f} {mt:>10.2f} {id_v:>10.4f}")

## Hick's Law

$RT = a + b \cdot \log_2(N + 1)$

Reaction time increases logarithmically with the number of choices.

In [ ]:
def hick_reaction_time(n_choices, a, b):
    """RT = a + b * log2(N + 1)"""
    return a + b * math.log2(n_choices + 1)

# Canonical: N=7 (weapon wheel with 7 weapons)
rt_7 = hick_reaction_time(7, HICK_A_MS, HICK_B_MS)
print(f"Hick RT (N=7): {rt_7:.10f} ms")

# Sweep: menu depth design implications
print(f"\n{'N choices':>10} {'RT (ms)':>10} {'Design implication'}")
print("-" * 50)
for n in [2, 4, 7, 10, 16, 32]:
    rt = hick_reaction_time(n, HICK_A_MS, HICK_B_MS)
    impl_note = "fast" if rt < 500 else "slow — consider grouping" if rt < 700 else "too many — split"
    print(f"{n:>10} {rt:>10.2f} {impl_note}")

## Steering Law

$T = a + b \cdot \frac{D}{W}$

Time to navigate through a tunnel of width W over distance D.

In [ ]:
def steering_time(distance, width, a, b):
    """T = a + b * (D/W)"""
    return a + b * (distance / width)

# Canonical: D=100, W=20, a=10, b=5
st = steering_time(100.0, 20.0, 10.0, 5.0)
print(f"Steering time (D=100, W=20): {st:.10f} ms")
assert abs(st - 35.0) < 1e-10, "Steering law sanity check failed"

## Validation Against Rust Implementation

These exact values are compiled into `ludoSpring/barracuda/src/tolerances/interaction.rs`
and verified by `cargo test`. Any change to the constants or formulas here must be
reflected in the Rust implementation.

In [ ]:
# Cross-check analytical values (these are the golden values in Rust)
expected_fitts = 50.0 + 150.0 * math.log2(2.0 * 100.0 / 10.0 + 1.0)
expected_hick = 200.0 + 150.0 * math.log2(8.0)
expected_steering = 35.0

checks = [
    ("fitts_mt", fitts_movement_time(100.0, 10.0, 50.0, 150.0), expected_fitts),
    ("hick_rt", hick_reaction_time(7, 200.0, 150.0), expected_hick),
    ("steering", steering_time(100.0, 20.0, 10.0, 5.0), expected_steering),
]

all_pass = True
for name, actual, expected in checks:
    ok = abs(actual - expected) < 1e-10
    status = "PASS" if ok else "FAIL"
    print(f"{name}: {status} (actual={actual:.15f}, expected={expected:.15f})")
    if not ok:
        all_pass = False

assert all_pass, "Baseline drift detected — update Rust golden values"
print("\nAll baselines PASS — Rust golden values are correct.")